In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

In [2]:
df = pd.read_excel("../data/get_around_delay_analysis.xlsx")
df.head()

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN


In [3]:
print(df.shape)
print(round(df.isna().sum() / df.shape[0], 2))
df.describe()

(21310, 7)
rental_id                                     0.00
car_id                                        0.00
checkin_type                                  0.00
state                                         0.00
delay_at_checkout_in_minutes                  0.23
previous_ended_rental_id                      0.91
time_delta_with_previous_rental_in_minutes    0.91
dtype: float64


,rental_id,car_id,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
count,21310.000000,21310.000000,16346.000000,1841.000000,1841.000000
mean,549712.880338,350030.603426,59.701517,550127.411733,279.288430
std,13863.446964,58206.249765,1002.561635,13184.023111,254.594486
min,504806.000000,159250.000000,-22433.000000,505628.000000,0.000000
25%,540613.250000,317639.000000,-36.000000,540896.000000,60.000000
50%,550350.000000,368717.000000,9.000000,550567.000000,180.000000
75%,560468.500000,394928.000000,67.000000,560823.000000,540.000000
max,576401.000000,417675.000000,71084.000000,575053.000000,720.000000


In [4]:
data_delay = df.copy()

data_delay['was_late'] = (
    (data_delay['delay_at_checkout_in_minutes'] > 0) & 
    (data_delay['state'] == 'ended')
    )

data_delay['rental_started_with_delay'] = (
    (data_delay['state'] == 'ended') &
    (data_delay['previous_ended_rental_id'].notna()) &
    (data_delay['delay_at_checkout_in_minutes'] > data_delay['time_delta_with_previous_rental_in_minutes'])
)

data_delay.head()

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes,was_late,rental_started_with_delay
0,505000,363965,mobile,canceled,NaN,NaN,NaN,False,False
1,507750,269550,mobile,ended,-81.0,NaN,NaN,False,False
2,508131,359049,connect,ended,70.0,NaN,NaN,True,False
3,508865,299063,connect,canceled,NaN,NaN,NaN,False,False
4,511440,313932,mobile,ended,NaN,NaN,NaN,False,False


In [5]:
print(f"Share of rentals returned late: {data_delay['was_late'].sum() / data_delay.shape[0]:.2%}")
print(f"Percentage of rental started with delay: {data_delay['rental_started_with_delay'].sum() / data_delay.shape[0]:.2%}")

Share of rentals returned late: 44.13%
Percentage of rental started with delay: 1.27%


In [6]:
data_delay[data_delay['state'] == 'ended']

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes,was_late,rental_started_with_delay
1,507750,269550,mobile,ended,-81.0,NaN,NaN,False,False
2,508131,359049,connect,ended,70.0,NaN,NaN,True,False
4,511440,313932,mobile,ended,NaN,NaN,NaN,False,False
5,511626,398802,mobile,ended,-203.0,NaN,NaN,False,False
6,511639,370585,connect,ended,-15.0,563782.0,570.0,False,False
...,...,...,...,...,...,...,...,...,...
21305,573446,380069,mobile,ended,NaN,573429.0,300.0,False,False
21306,573790,341965,mobile,ended,-337.0,NaN,NaN,False,False
21307,573791,364890,mobile,ended,144.0,NaN,NaN,True,False
21308,574852,362531,connect,ended,-76.0,NaN,NaN,False,False


In [7]:
data_delay_ended = data_delay[data_delay['state'] == 'ended']

print(f"Share of rentals returned late (ended only): {data_delay_ended['was_late'].sum() / data_delay_ended.shape[0]:.2%}")
print(f"Percentage of rental started with delay: {data_delay_ended['rental_started_with_delay'].sum() / data_delay_ended.shape[0]:.2%}")

Share of rentals returned late (ended only): 52.11%
Percentage of rental started with delay: 1.50%


In [21]:
print(round(data_delay_ended['previous_ended_rental_id'].notna().sum() / data_delay_ended.shape[0], 2))
print(data_delay_ended['previous_ended_rental_id'].notna().sum())

0.09
1612


In [36]:
data_delay_at_risk = data_delay_ended[data_delay_ended['previous_ended_rental_id'].notna()]
print(f"Percentage of rentals who started with delay: {(data_delay_at_risk['rental_started_with_delay'] == True).sum() / data_delay_at_risk.shape[0]:.2%}")
print(f"Number of rentals who started with delay: {(data_delay_at_risk['rental_started_with_delay'] == True).sum()}")

Percentage of rentals who started with delay: 16.75%
Number of rentals who started with delay: 270


## How often are drivers late for the next check-in? How does it impact the next driver?

To answer this question properly, we focused only on rentals that were actually completed (`state == "ended"`).  
Canceled rentals were excluded since they do not generate a checkout and therefore cannot produce a delay.  

52.11% of completed rentals are returned late.  
However, only 1.50% of completed rentals actually start late due to a previous late return.  
Additionally, only 9% of completed rentals even have a previous booking, meaning that only a small subset of rentals is structurally exposed to an overlap risk.  
Among this at-risk subset, 16.75% of rentals result in an actual overlap, meaning that nearly 1 out of 6 consecutive rentals leads to a real conflict.  

Late returns are very common, more than one out of two rentals is returned after the scheduled checkout time.  
However, the real operational impact on the next driver remains limited at a global scale, since only a small portion of rentals are both consecutive and delayed beyond the available buffer.  
When focusing specifically on consecutive bookings, the risk becomes significantly higher.  
While overlaps represent only 1.50% of all completed rentals, they account for 16.75% of cases where a rental is immediately followed by another.  

This indicates that delays are not evenly distributed across the platform:  
they are highly concentrated within short back-to-back bookings, which represent the critical risk zone for customer friction.  

In [37]:
data_delay_at_risk.head()

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes,was_late,rental_started_with_delay
6,511639,370585,connect,ended,-15.0,563782.0,570.0,False,False
19,519491,312389,mobile,ended,58.0,545639.0,420.0,True,False
23,521156,392479,mobile,ended,NaN,537298.0,0.0,False,False
34,525044,349751,mobile,ended,NaN,510607.0,60.0,False,False
40,528808,181625,connect,ended,-76.0,557404.0,330.0,False,False


In [38]:
(data_delay_at_risk['rental_started_with_delay'] == True).mean(data_delay_at_risk['time_delta_with_previous_rental_in_minutes'])

TypeError: unhashable type: 'Series'